# SVD with a small picture

I build the example one step at a time. Run the cells in order. Install NumPy and Matplotlib first. The brightness values and errors are dimensionless.

[Read the picture explanation](https://bjoyita.github.io/posts/svd-small-picture.html).

## 1. Import the libraries

NumPy performs the array calculations. Matplotlib displays the images. I use the short names np and plt so the later lines are easier to read. Install them with python -m pip install numpy matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Create the small picture

I start with an 8 × 8 array whose brightness is 0.10. The following assignments draw the roof, walls, windows and door. In image[1, 3:5], 1 selects the second row; 3:5 selects columns 3 and 4 because the stopping index is excluded. Brightness is dimensionless, with 0 for black and 1 for white.

In [ ]:
image = np.full((8, 8), 0.10)
image[1, 3:5] = 0.75
image[2, 2:6] = 0.75
image[3, 1:7] = 0.75
image[4:7, 2:6] = 0.50
image[4, 2] = image[4, 5] = 0.80
image[5:7, 3:5] = 0.20

## 3. Calculate the SVD

The calculation returns three arrays. U has eight rows and eight columns. strengths contains eight singular values. Vt has eight rows and eight columns. A column of U describes variation down the image rows, and the matching row of Vt describes variation across its columns. The matching singular value sets the strength. Only four layers are needed for this constructed picture.

In [ ]:
U, strengths, Vt = np.linalg.svd(image, full_matrices=False)

## 4. Add the layers and calculate the error

range(5) gives 0, 1, 2, 3 and 4. The slice U[:, :count] keeps every row and the first count columns. strengths[:count] keeps the corresponding strengths. Multiplication scales each retained column by its strength. The operator @ then performs matrix multiplication with the matching rows of Vt. With count = 0, the product is a zero image. I store each result in reconstructions. The RMSE compares corresponding pixels, squares their differences, averages them and takes the square root.

In [ ]:
reconstructions = []
for count in range(5):
    rebuilt = (U[:, :count] * strengths[:count]) @ Vt[:count]
    reconstructions.append(rebuilt)
    rmse = np.sqrt(np.mean((image - rebuilt)**2))
    print(f"Layers: {count}, brightness RMSE: {rmse:.6f}")

## 5. Check the result

np.allclose checks agreement while allowing small floating-point rounding differences. I check that four layers recover the original image. The list named errors contains mean squared errors for all five reconstructions. The next check confirms that adding a retained SVD layer does not increase this error. These checks concern this particular example.

In [ ]:
assert np.allclose(reconstructions[4], image)
errors = [np.mean((image - rebuilt)**2) for rebuilt in reconstructions]
assert all(errors[i + 1] <= errors[i] for i in range(4))

## 6. Display the pictures

plt.subplots(1, 5) creates one row of five plotting areas. enumerate supplies both a layer count and a plotting area. I keep vmin = 0 and vmax = 1 in every panel so their brightness values can be compared. interpolation = nearest keeps pixel edges sharp. Removing the tick labels makes this small comparison easier to see. tight_layout adjusts the spacing; show opens the figure. When running the script in a desktop window, close the figure to reach the final confirmation message.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for count, ax in enumerate(axes):
    ax.imshow(reconstructions[count], cmap="gray", vmin=0, vmax=1,
              interpolation="nearest")
    ax.set_title(f"{count} layers")
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()
print("All picture checks passed.")

## Expected output

The printed RMSE values, rounded to six decimal places, are 0.397060, 0.139208, 0.092277, 0.024436 and 0.000000 for zero through four layers. The last value is smaller than the displayed precision, rather than a guarantee of exact arithmetic. The final message is “All picture checks passed.”

Try changing the number of retained layers first. If you change the picture itself, it may need more than four layers; review the checks as well.